# E-COMMERCE BUSINESS ANALYSIS

# 1. Project Introduction

## Business Context

The goal of this project is to analyze an e-commerce dataset in order to better understand customer behavior, marketing performance, 
product performance, and sales activity.

The analysis focuses on identifying:
- what generates the most revenue,
- which customer acquisition channels perform best,
- where users may leave before purchasing,
- and what business opportunities or problems can be detected from the data.

This project also aims to explore:
- customer conversion behavior,
- loyalty and repeat purchasing,
- marketing efficiency,
- and the impact of different experiment groups on user behavior.


---

# 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---

# 3. Load Datasets

In [ ]:
campaigns_df = pd.read_csv(r"C:\Users\Tooba Shaukat\OneDrive\ecommerce\campaigns.csv", index_col=0)

In [ ]:
campaigns_df.head()

In [ ]:
campaigns_df.shape

In [ ]:
campaigns_df.columns

In [ ]:
customers_df = pd.read_csv(r"C:\Users\Tooba Shaukat\OneDrive\ecommerce\customers.csv", index_col=0)

In [ ]:
customers_df.head()

In [ ]:
customers_df.shape

In [ ]:
customers_df.columns

In [ ]:
events_df = pd.read_csv(r"C:\Users\Tooba Shaukat\OneDrive\ecommerce\events.csv", index_col=0)

In [ ]:
events_df.head()

In [ ]:
events_df.shape

In [ ]:
events_df.columns

In [ ]:
products_df = pd.read_csv(r"C:\Users\Tooba Shaukat\OneDrive\ecommerce\products.csv", index_col=0)

In [ ]:
products_df.head()

In [ ]:
products_df.shape

In [ ]:
products_df.columns

In [ ]:
transactions_df = pd.read_csv(r"C:\Users\Tooba Shaukat\OneDrive\ecommerce\transactions.csv", index_col=0)

In [ ]:
transactions_df.head()

In [ ]:
transactions_df.shape

In [ ]:
transactions_df.columns

Dataset	             Rows	               Role

campaigns	         50	                   Marketing strategy
customers	         100,000	           Customer profile
events	             2,000,000	           Website behavior
products	         2,000	               Product catalog
transactions	     103,127	           Revenue / purchases

---


# 4. Dataset Relationships & Data Model

The analysis is based on five interconnected datasets representing different aspects of an e-commerce business ecosystem:

## 1. campaigns_df
Contains marketing campaign information:
- campaign channels,
- objectives,
- target segments,
- expected uplift.

Primary key:
- `campaign_id`

---

## 2. customers_df
Contains customer demographic and acquisition information:
- country,
- age,
- gender,
- loyalty tier,
- acquisition channel.

Primary key:
- `customer_id`

---

## 3. events_df
Tracks customer behavioral interactions on the platform:
- views,
- clicks,
- add-to-cart events,
- purchases,
- bounce events.

This table connects:
- customers,
- products,
- and campaigns.

Foreign keys:
- `customer_id`
- `product_id`
- `campaign_id`

---

## 4. products_df
Contains product catalog information:
- category,
- brand,
- base price,
- premium status.

Primary key:
- `product_id`

---

## 5. transactions_df
Contains completed transaction information:
- purchased products,
- revenue,
- discounts,
- refunds,
- campaign attribution.

Foreign keys:
- `customer_id`
- `product_id`
- `campaign_id`

---

## Dataset Relationships

The datasets are connected through shared identifiers:

- `customer_id`
  links customers with events and transactions.

- `product_id`
  links products with events and transactions.

- `campaign_id`
  links marketing campaigns with events and transactions.

These relationships make it possible to analyze:
- customer behavior,
- marketing performance,
- product performance,
- conversion funnels,
- and revenue generation across the e-commerce ecosystem.

---


# 5. Data Quality Checks

In [ ]:
for name, df in {
    "campaigns": campaigns_df,
    "customers": customers_df,
    "events": events_df,
    "products": products_df,
    "transactions": transactions_df
}.items():
    print("\n", name.upper())
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum())

## Investigating duplicate values in customers_df

In [ ]:
customers_df[customers_df.duplicated()].head()

In [ ]:
duplicate_rows = customers_df[
    customers_df.duplicated(keep=False)
]

duplicate_rows.sort_index().head(10)

In [ ]:
customers_df.index.duplicated().sum()

### Duplicate Analysis

Initial duplicate detection suggested repeated rows in the customer dataset.

However, further investigation showed that these rows correspond to different customer IDs sharing similar demographic profiles rather than true duplicate customers.

Therefore, no rows were removed from the customer dataset.

## Investigating missing values in events_df

In [ ]:
missing_product_by_event = (
    events_df
    .groupby("event_type")["product_id"]
    .apply(lambda x: x.isna().mean() * 100)
    .sort_values(ascending=False)
)

missing_product_by_event

## Missing Product ID Analysis

- Bounce events show 100% missing product identifiers, which is expected because users may leave the website before interacting with a specific product.

- Product-related interactions such as views, clicks, and add-to-cart events have complete product tracking, indicating strong tracking consistency for browsing behavior.

- However, approximately 10% of purchase events are missing product identifiers. This may indicate incomplete transaction attribution or tracking inconsistencies during the purchase process.

This issue could reduce the accuracy of:
- product performance analysis,
- campaign attribution,
- and revenue reporting.

## Filling missing values in device_type

In [ ]:
events_df["device_type"].value_counts(dropna=False)

In [ ]:
events_df["device_type"] = events_df["device_type"].fillna("Unknown")

In [ ]:
events_df["device_type"].value_counts(dropna=False)

## Device Distribution Analysis

Mobile traffic represents the majority of website interactions, suggesting that mobile experience optimization is likely critical for business performance.

Desktop traffic remains significant, while tablet usage is comparatively low.

A small proportion of sessions have unknown device information, which may result from incomplete tracking or unidentified traffic sources.

## Investigating missing values in transactions_df

In [ ]:
transactions_df[
    transactions_df["product_id"].isna()
].head()

In [ ]:
transactions_df[
    transactions_df["gross_revenue"].isna()
].head()

In [ ]:
transactions_df[
    transactions_df["product_id"].isna() &
    transactions_df["gross_revenue"].isna()
].shape

 Analysis: All rows with missing product_id
also have missing gross_revenue

In [ ]:
missing_transaction_rate = (
    transactions_df["gross_revenue"].isna().mean() * 100
)

missing_transaction_rate

In [ ]:
transactions_df[
    transactions_df["gross_revenue"].isna()
]["campaign_id"].value_counts().head(10)

## Transaction Data Quality & Revenue Leakage

Approximately 10.13% of transaction records contain missing product and revenue information.

The issue is not isolated to a single campaign and appears across multiple marketing campaigns.

Campaign ID 0 accounts for the highest number of incomplete transactions, which may indicate weaker attribution tracking for unattributed or direct traffic sources.

Potential business impacts include:
- inaccurate campaign ROI measurement,
- underestimation of revenue,
- reduced product-level visibility,
- and weaker conversion analysis.

This may represent a form of operational or analytical revenue leakage.

## CONVERT DATES

In [ ]:
campaigns_df["start_date"] = pd.to_datetime(campaigns_df["start_date"])
campaigns_df["end_date"] = pd.to_datetime(campaigns_df["end_date"])
customers_df["signup_date"] = pd.to_datetime(customers_df["signup_date"])
events_df["timestamp"] = pd.to_datetime(events_df["timestamp"])
products_df["launch_date"] = pd.to_datetime(products_df["launch_date"])
transactions_df["timestamp"] = pd.to_datetime(transactions_df["timestamp"])

## TRAFFIC SOURCE CLEANING

During the analysis of traffic sources, some categories appeared duplicated because of inconsistent capitalization.

For example:
- "EMAIL" and "Email"
- "SOCIAL" and "Social"
- "ORGANIC" and "Organic"

This type of inconsistency can affect grouping and aggregation results during analysis.

To ensure accurate analysis, traffic source values were standardized using title formatting.

In [ ]:
events_df["traffic_source"].value_counts()

In [ ]:
events_df["traffic_source"] = (
    events_df["traffic_source"]
    .str.title()
)

In [ ]:
events_df["traffic_source"].value_counts()

---


# 7. Data Preparation for Analysis
### Creating Clean Transactions Dataset

Removing incomplete transactions.

The transactions dataset contains incomplete rows with missing:
- product identifiers,
- and revenue values.

Since these transactions cannot be used reliably for revenue analysis, they are excluded from KPI and business analysis calculations.

In [ ]:
# creating clean transactions dataset
transactions_clean = transactions_df.dropna(
    subset=["gross_revenue", "product_id"]
)

In [ ]:
# verifying missing values removed
transactions_clean.isna().sum()

The cleaned transaction dataset will be used for revenue, customer, and product performance analysis.

## Creating Merged Analytical Tables

To perform deeper business analysis, the cleaned transaction dataset is merged with:
- the product dataset,
- and the customer dataset.

This allows analysis of:
- product performance,
- customer behavior,
- acquisition channels,
- loyalty segments,
- and revenue generation.

In [ ]:
#Merge Transactions + Products
transactions_products = transactions_clean.merge(
    products_df,
    on="product_id",
    how="left"
)

In [ ]:
transactions_products.head()

This merged dataset combines transaction and product information, enabling category-level and product-level revenue analysis.

In [ ]:
# Merge Transactions + Customers
transactions_customers = transactions_clean.merge(
    customers_df,
    on="customer_id",
    how="left"
)

In [ ]:
transactions_customers.head()

This merged dataset combines transaction and customer information, enabling analysis of:
- acquisition channels,
- loyalty tiers,
- customer value,
- and purchasing behavior.

---


# 8. Business KPIs

In [ ]:
# Core KPIs

total_revenue = transactions_clean["gross_revenue"].sum()

num_transactions = transactions_clean.shape[0]

avg_order_value = transactions_clean["gross_revenue"].mean()

refund_rate = (
    transactions_clean["refund_flag"].mean() * 100
)

total_customers = (
    transactions_clean["customer_id"].nunique()
)

print(f"Total Revenue: ${total_revenue:,.2f}")

print(f"Number of Transactions: {num_transactions:,}")

print(f"Average Order Value: ${avg_order_value:.2f}")

print(f"Refund Rate: {refund_rate:.2f}%")

print(f"Unique Customers: {total_customers:,}")

In [ ]:
## KPI	Value

Total Revenue	             $8.37M
Transactions	             92,678
AOV	                         $90.36
Refund Rate	                 2.92%
Purchase Rate	             5.16%
Bounce Rate	                 9.50%

## Initial KPI Insights

The e-commerce platform generated more than $8.3M in revenue across approximately 92k completed transactions.

The Average Order Value (~$90) suggests relatively healthy basket sizes and may indicate meaningful contribution from higher-value products or premium categories.

The refund rate remains below 3%, which may reflect relatively stable operational performance and acceptable customer satisfaction levels.

The relationship between transaction volume and unique customers also suggests the presence of repeat purchasing behavior, which is an important indicator for customer retention and long-term value.

---


# 9. Product Category Analysis

## Revenue by Product Category

In [ ]:
transactions_products = transactions_clean.merge(
    products_df,
    on="product_id",
    how="left"
)

In [ ]:
category_revenue = (
    transactions_products
    .groupby("category")["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
)

category_revenue

In [ ]:
category_revenue.plot(kind="bar")

plt.title("Revenue by Product Category")

plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.show()

## Revenue by Category Insights

Electronics is the dominant revenue-generating category, contributing substantially more revenue than all other product categories.

This may reflect:
- higher product pricing,
- stronger customer demand,
- or the contribution of premium products.

Home products also represent a significant revenue contributor, while Grocery and Beauty generate comparatively limited revenue.

Lower-performing categories may represent:
- underexploited opportunities,
- weaker conversion performance,
- or lower customer demand.

### Refund Rate by Product Category

In [ ]:
refund_by_category = (
    transactions_products
    .groupby("category")["refund_flag"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

refund_by_category

## Refund Rate by Category Insights

Refund rates remain relatively stable across product categories, ranging between approximately 2.7% and 3.2%.

Sports and Beauty categories show the highest refund rates, which may indicate:
- product expectation mismatches,
- sizing or quality issues,
- or weaker customer satisfaction.

Although Electronics does not have the highest refund percentage, it represents the most financially significant category due to its dominant revenue contribution.

As a result, even moderate refund activity within Electronics may have a meaningful impact on profitability and revenue retention.

---


# 10. Acquisition Channel Analysis

### Revenue by Acquisition Channel

In [ ]:
channel_revenue = (
    transactions_customers
    .groupby("acquisition_channel")["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
)

channel_revenue

## Revenue by Acquisition Channel Insights

Organic and Paid Search channels generate the highest revenue contribution, suggesting that search-driven acquisition plays a major role in overall business performance.

Strong Organic performance may indicate:
- effective SEO visibility,
- strong brand awareness,
- or high-intent customer traffic.

Paid Search also contributes significantly to revenue, suggesting that paid acquisition campaigns are effective at driving monetizable traffic.

Email marketing generates meaningful revenue as well, which may reflect successful retention and reactivation strategies targeting existing customers.

Referral remains the lowest-performing acquisition channel and may represent either:
- a lower strategic priority,
- or a potential underdeveloped growth opportunity.

## Average Revenue Per Customer by Acquisition Channel

In [ ]:
revenue_per_customer = (
    transactions_customers
    .groupby("acquisition_channel")
    .agg(
        total_revenue=("gross_revenue", "sum"),
        unique_customers=("customer_id", "nunique")
    )
)

revenue_per_customer["revenue_per_customer"] = (
    revenue_per_customer["total_revenue"] /
    revenue_per_customer["unique_customers"]
)

revenue_per_customer.sort_values(
    by="revenue_per_customer",
    ascending=False
)

## Revenue per Customer Insights

While Organic traffic generates the highest total revenue, Email customers generate the highest revenue per customer.

This suggests that retention and CRM-driven acquisition strategies may produce particularly valuable customers with stronger purchasing behavior.

Organic traffic combines both:
- high total revenue,
- and strong customer value,

making it one of the most strategically important acquisition channels.

Social acquisition generates comparatively lower revenue per customer, which may indicate lower purchase intent or weaker conversion quality.

---

# 11. Loyalty Tier Analysis

### Revenue by Loyalty Tier

In [ ]:
loyalty_revenue = (
    transactions_customers
    .groupby("loyalty_tier")
    .agg(
        total_revenue=("gross_revenue", "sum"),
        unique_customers=("customer_id", "nunique"),
        avg_order_value=("gross_revenue", "mean")
    )
    .sort_values(by="total_revenue", ascending=False)
)

loyalty_revenue

### Loyalty Tier Insights

Bronze customers generate the highest overall revenue contribution, primarily due to their large customer base.

However, Silver customers exhibit the highest Average Order Value, suggesting stronger purchasing behavior on a per-transaction basis.

Interestingly, Gold and Platinum customers do not demonstrate substantially higher order values compared to lower tiers. This may indicate:
- limited differentiation in customer spending behavior across loyalty levels,
- or opportunities to strengthen the effectiveness of the loyalty program.

The relatively small Platinum customer base may also suggest an opportunity to improve customer progression toward higher loyalty tiers.

### Average Order Value by Loyalty Tier

In [ ]:
aov_loyalty = (
    transactions_customers
    .groupby("loyalty_tier")["gross_revenue"]
    .mean()
    .sort_values(ascending=False)
)

aov_loyalty

In [ ]:
aov_loyalty.plot(kind="bar")

plt.title("Average Order Value by Loyalty Tier")

plt.ylabel("Average Order Value ($)")

plt.xticks(rotation=0)

plt.show()

### Average Order Value by Loyalty Tier Insights

Silver customers show the highest Average Order Value, meaning they spend slightly more per transaction compared to other loyalty tiers.

Gold and Platinum customers do not show significantly higher order values, but earlier analysis showed that they purchase more frequently.

This suggests that higher loyalty tiers may influence repeat purchasing behavior more than basket size.

### Transactions per Customer by Loyalty Tier

In [ ]:
loyalty_frequency = (
    transactions_customers
    .groupby("loyalty_tier")
    .agg(
        transactions=("customer_id", "count"),
        unique_customers=("customer_id", "nunique")
    )
)

loyalty_frequency["transactions_per_customer"] = (
    loyalty_frequency["transactions"] /
    loyalty_frequency["unique_customers"]
)

loyalty_frequency.sort_values(
    by="transactions_per_customer",
    ascending=False
)

## Loyalty Frequency Insights

Gold and Platinum customers demonstrate the highest transaction frequency per customer, suggesting that higher loyalty tiers are associated with stronger repeat purchasing behavior.

Interestingly, these higher-tier customers do not exhibit substantially higher Average Order Values compared to lower tiers. This suggests that the loyalty program may currently influence purchase frequency more effectively than basket size.

Bronze customers generate the largest overall revenue contribution due to their large population, but they show the lowest transaction frequency, indicating lower long-term engagement.

Silver customers represent an especially interesting segment because they combine:
- the highest Average Order Value,
- with moderate purchase frequency.

This may represent a strong opportunity for targeted retention and upsell strategies aimed at moving Silver customers toward higher loyalty engagement.

---

# 12. Conversion Funnel Analysis
where customers are lost before generating revenue.

### Counting each event type

In [ ]:
event_counts = (
    events_df["event_type"]
    .value_counts()
)

event_counts

### Calculating percentages

In [ ]:
event_percentages = (
    events_df["event_type"]
    .value_counts(normalize=True) * 100
)

event_percentages

### Creating funnel visualization

In [ ]:
event_counts.plot(kind="bar")

plt.title("Customer Funnel Events")

plt.ylabel("Number of Events")

plt.xticks(rotation=45)

plt.show()

### Calculating Purchase Rate

In [ ]:
purchase_rate = (
    (events_df["event_type"] == "purchase").mean() * 100
)

purchase_rate

### Calculating Bounce Rate

In [ ]:
bounce_rate = (
    (events_df["event_type"] == "bounce").mean() * 100
)

bounce_rate

## Conversion Funnel Insights

The conversion funnel analysis highlights how users progress from website interaction to completed purchases.

The purchase rate is approximately 5.16%, meaning that only a relatively small proportion of user events ultimately result in a purchase. This is expected in e-commerce environments where many users browse products without completing transactions.

The bounce rate is approximately 9.50%, indicating that a noticeable proportion of users leave the platform quickly without meaningful interaction. This may suggest:
- low-intent traffic,
- weak landing page engagement,
- or early-stage user friction.

The gap between browsing activity and completed purchases suggests potential conversion drop-off during the customer journey. Possible explanations may include:
- checkout friction,
- pricing sensitivity,
- shipping costs,
- payment issues,
- or product decision hesitation.

Improving conversion efficiency within the funnel could represent an important opportunity for increasing revenue without necessarily increasing traffic acquisition costs.

---

# 13. Device Conversion Analysis

### Analysing event distribution by device

In [ ]:
device_events = pd.crosstab(
    events_df["device_type"],
    events_df["event_type"]
)

device_events

### Calculating purchase rate by device

In [ ]:
purchase_by_device = (
    events_df
    .groupby("device_type")
    .apply(
        lambda x: (x["event_type"] == "purchase").mean() * 100
    )
    .sort_values(ascending=False)
)

purchase_by_device

### Calculating bounce rate by device

In [ ]:
bounce_by_device = (
    events_df
    .groupby("device_type")
    .apply(
        lambda x: (x["event_type"] == "bounce").mean() * 100
    )
    .sort_values(ascending=False)
)

bounce_by_device

### Visualising purchase rate by device

In [ ]:
purchase_by_device.plot(kind="bar")

plt.title("Purchase Rate by Device")

plt.ylabel("Purchase Rate (%)")

plt.xticks(rotation=45)

plt.show()

## Device Conversion Insights

Despite mobile representing the majority of platform traffic, conversion rates remain remarkably consistent across all device types.

Purchase rates are nearly identical between:
- mobile,
- desktop,
- and tablet users.

Bounce rates also remain highly stable across devices, suggesting that the platform currently provides a relatively consistent user experience regardless of device type.

This indicates that device-specific friction does not appear to be a major conversion issue at a high level.

---

# 14. Traffic Source Conversion Analysis

### Purchase rate by traffic source

In [ ]:
purchase_by_source = (
    events_df
    .groupby("traffic_source")["event_type"]
    .apply(lambda x: (x == "purchase").mean() * 100)
    .sort_values(ascending=False)
)

purchase_by_source

### Bounce rate by traffic source

In [ ]:
bounce_by_source = (
    events_df
    .groupby("traffic_source")["event_type"]
    .apply(lambda x: (x == "bounce").mean() * 100)
    .sort_values(ascending=False)
)

bounce_by_source

### Visualising purchase rate

In [ ]:
purchase_by_source.plot(kind="bar")

plt.title("Purchase Rate by Traffic Source")

plt.ylabel("Purchase Rate (%)")

plt.xticks(rotation=45)

plt.show()

### Visualising bounce rate

In [ ]:
bounce_by_source.plot(kind="bar")

plt.title("Bounce Rate by Traffic Source")

plt.ylabel("Bounce Rate (%)")

plt.xticks(rotation=45)

plt.show()

## Traffic Source Conversion Insights

Email traffic demonstrates the highest purchase conversion rate, suggesting that CRM and retention-driven traffic sources generate highly engaged and high-intent users.

Paid Search also performs strongly, indicating effective acquisition targeting and strong commercial intent among paid users.

Organic traffic generates substantial overall revenue but exhibits comparatively low purchase conversion rates. This suggests that Organic channels may attract broader exploratory traffic with lower immediate purchase intent.

Bounce rates remain relatively stable across traffic sources, although Organic and Direct traffic show slightly higher bounce behavior compared to Email and Paid Search.

Overall, the analysis suggests that:
- Email and Paid Search deliver the strongest conversion efficiency,
- while Organic traffic plays a major role in top-of-funnel acquisition and long-term traffic generation.

---

# 15. A/B Testing Analysis

### Counting events by experiment group

In [ ]:
experiment_counts = pd.crosstab(
    events_df["experiment_group"],
    events_df["event_type"]
)

experiment_counts

### Purchase rate by experiment group

In [ ]:
purchase_by_experiment = (
    events_df
    .groupby("experiment_group")["event_type"]
    .apply(lambda x: (x == "purchase").mean() * 100)
    .sort_values(ascending=False)
)

purchase_by_experiment

### Bounce rate by experiment group

In [ ]:
bounce_by_experiment = (
    events_df
    .groupby("experiment_group")["event_type"]
    .apply(lambda x: (x == "bounce").mean() * 100)
    .sort_values(ascending=False)
)

bounce_by_experiment

### Session duration by experiment group

In [ ]:
session_duration_experiment = (
    events_df
    .groupby("experiment_group")["session_duration_sec"]
    .mean()
    .sort_values(ascending=False)
)

session_duration_experiment

### Visualising purchase rate

In [ ]:
purchase_by_experiment.plot(kind="bar")

plt.title("Purchase Rate by Experiment Group")

plt.ylabel("Purchase Rate (%)")

plt.xticks(rotation=0)

plt.show()

## A/B Testing Insights

Variant_B demonstrates the strongest overall performance across the experiment groups.

Compared to the Control group:
- Variant_B achieves the highest purchase conversion rate,
- while also slightly reducing bounce rate.

This suggests that Variant_B may provide a more effective or less frictional user experience during the customer journey.

Session duration remains relatively stable across all experiment groups, indicating that the improvement in conversion performance is not driven by longer browsing sessions but potentially by more efficient user interaction or improved purchase flow.

Overall, Variant_B appears to be the most effective experiment variation and may represent the strongest candidate for broader deployment.

---

# 16. Session Duration & Purchase Behavior Analysis

Do purchasing users behave differently from bouncing users?

### Average session duration by event type

In [ ]:
session_by_event = (
    events_df
    .groupby("event_type")["session_duration_sec"]
    .mean()
    .sort_values(ascending=False)
)

session_by_event

In [ ]:
session_by_event.plot(kind="bar")

plt.title("Average Session Duration by Event Type")

plt.ylabel("Average Session Duration (sec)")

plt.xticks(rotation=45)

plt.show()

## Session Duration Insights

Average session duration remains relatively stable across all event types, with only minor differences between purchasing, browsing, and bounce behavior.

This suggests that session duration alone may not strongly explain conversion behavior within the platform.

Purchasing users do not appear to spend substantially more time on the platform than non-purchasing users, which may indicate:
- relatively fast purchase decision-making,
- efficient navigation behavior,
- or the influence of other conversion drivers such as traffic quality, pricing, or user intent.

---

# 17. Final Insights And Recommendations

## Key Findings

- Electronics is the strongest revenue-generating category.
- Email traffic shows the highest purchase conversion rate.
- Organic traffic generates high revenue but lower conversion efficiency.
- Gold and Platinum customers purchase more frequently.
- Variant_B achieved the strongest conversion performance.
- Around 10% of transactions appear incomplete and may indicate tracking or checkout issues.

---

## Business Recommendations



## 1. Strengthen Email and Paid Search strategies

Email shows the highest purchase conversion rate, while Paid Search also performs strongly.  
These channels appear to bring high-intent users and should be prioritized for conversion-focused campaigns.

## 2. Maintain Organic as a key acquisition channel

Organic generates strong revenue volume, but its purchase conversion rate is lower than Email and Paid Search.  
This suggests that Organic is valuable for traffic generation and brand visibility, but visitors may need better nurturing before purchase.

## 3. Scale Variant_B experience

Variant_B achieved the highest purchase rate and the lowest bounce rate among experiment groups.  
This version appears to improve conversion efficiency and should be considered for broader deployment.

## 4. Improve tracking of incomplete transactions

Around 10% of transactions contain missing product and revenue information.  
This may indicate failed transactions, incomplete checkout tracking, or revenue attribution issues.

This should be investigated because it can affect:
- revenue reporting,
- campaign performance analysis,
- and product-level profitability analysis.

## 5. Optimize loyalty progression

Gold and Platinum customers purchase more frequently, while Silver customers show the highest average order value.  
A strong opportunity would be to encourage Silver customers to become more engaged and move toward higher loyalty tiers.

## 6. Monitor category-level performance

Electronics is the strongest revenue-driving category, while Sports and Beauty show the highest refund rates.  
Although refund rates are not alarming, high-revenue categories like Electronics should be monitored closely because even moderate refunds can have a larger financial impact.

---
    
## Priority KPIs to monitor weekly

- Total revenue
- Average Order Value
- Purchase conversion rate
- Bounce rate
- Refund rate
- Revenue by acquisition channel
- Revenue by product category
- Revenue per customer
- Transactions per customer
- Incomplete transaction rate

---

## Potential Next Steps

- Build a real-time dashboard for KPI monitoring.
- Further investigate incomplete transactions.
- Analyze customer retention over time.
- Explore campaign ROI if marketing spend data becomes available.

---

## Exporting Clean Datasets

In [ ]:
transactions_products.to_csv(
    "transactions_products.csv",
    index=False
)

transactions_customers.to_csv(
    "transactions_customers.csv",
    index=False
)

## Creating a reduced events dataset for looker studio

In [ ]:
events_dashboard = events_df[[
    "event_type",
    "device_type",
    "traffic_source",
    "experiment_group",
    "session_duration_sec"
]]

In [ ]:
events_dashboard.to_csv(
    "events_dashboard.csv",
    index=False
)

---